# Notebook 54 — Índice administrado con Databricks AI Search

Este notebook crea un índice **Delta Sync** sobre el corpus Silver.

Configuración requerida por el ejercicio:

- cliente canónico `databricks-ai-search`;
- endpoint `STANDARD`, reutilizando el único endpoint permitido en Free Edition;
- `pipeline_type="TRIGGERED"`;
- clave primaria `chunk_id`;
- texto `chunk_text` mediante `embedding_source_column`;
- embeddings administrados por Databricks con `databricks-gte-large-en`;
- espera activa hasta estado `ready` / `ONLINE`.

No usamos Direct Vector Access: no está disponible en Free Edition y, además, obligaría a
calcular y mantener embeddings manualmente.


## 1. Instalar el SDK canónico


In [ ]:
%pip install -q -U databricks-ai-search
%restart_python


## 2. Configuración


In [ ]:
import re
import time

dbutils.widgets.text("catalogo", "big_data_ii_2025", "1. Catálogo UC")
dbutils.widgets.text("esquema", "spark_examples", "2. Schema UC")
dbutils.widgets.text("endpoint_ai_search", "agenteval_ai_search", "3. Endpoint AI Search")
dbutils.widgets.text("timeout_min", "35", "4. Timeout del índice (min)")

CATALOG = dbutils.widgets.get("catalogo").strip()
SCHEMA = dbutils.widgets.get("esquema").strip()
PREFERRED_ENDPOINT = dbutils.widgets.get("endpoint_ai_search").strip()
TIMEOUT_MIN = int(dbutils.widgets.get("timeout_min"))

for nombre, valor in {
    "catalogo": CATALOG,
    "esquema": SCHEMA,
    "endpoint_ai_search": PREFERRED_ENDPOINT,
}.items():
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_-]*", valor):
        raise ValueError(f"Identificador inválido en {nombre}: {valor!r}")

T_CORPUS = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_corpus"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.agenteval_squadv2_index"
EMBEDDING_MODEL = "databricks-gte-large-en"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
assert spark.catalog.tableExists(T_CORPUS), (
    f"No existe {T_CORPUS}. Ejecuta primero los notebooks 52 y 53."
)

print(f"Corpus     : {T_CORPUS}")
print(f"Índice     : {INDEX_NAME}")
print(f"Embeddings : {EMBEDDING_MODEL}")


## 3. Confirmar o reactivar CDF

Este control es deliberadamente redundante. Una reescritura o cambio de propiedades entre
clases no debe dejar el índice sin su prerrequisito.


In [ ]:
props = {
    r["key"]: r["value"]
    for r in spark.sql(f"SHOW TBLPROPERTIES {T_CORPUS}").collect()
}
cdf_enabled = props.get("delta.enableChangeDataFeed", "false").lower() == "true"

if not cdf_enabled:
    print("CDF estaba deshabilitado; se habilitará nuevamente.")
    spark.sql(
        f"ALTER TABLE {T_CORPUS} "
        "SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
    )

props_after = {
    r["key"]: r["value"]
    for r in spark.sql(f"SHOW TBLPROPERTIES {T_CORPUS}").collect()
}
assert props_after.get("delta.enableChangeDataFeed", "false").lower() == "true"

print(f"✓ CDF habilitado en {T_CORPUS}")
print(f"✓ Filas del corpus: {spark.table(T_CORPUS).count():,}")


## 4. Crear o reutilizar el endpoint

Free Edition permite un solo endpoint de AI Search con una unidad. Si ya existe uno, lo
reutilizamos aunque tenga otro nombre. No borramos recursos automáticamente.


In [ ]:
from databricks.ai_search.client import AISearchClient

client = AISearchClient(disable_notice=True)
response = client.list_endpoints()

if isinstance(response, dict):
    endpoint_items = response.get("endpoints", [])
else:
    endpoint_items = list(response or [])

endpoint_names = [
    e.get("name") if isinstance(e, dict) else getattr(e, "name", None)
    for e in endpoint_items
]
endpoint_names = [name for name in endpoint_names if name]

if PREFERRED_ENDPOINT in endpoint_names:
    VS_ENDPOINT = PREFERRED_ENDPOINT
    print(f"Reutilizando el endpoint configurado: {VS_ENDPOINT}")
elif endpoint_names:
    VS_ENDPOINT = endpoint_names[0]
    print(
        f"Free Edition ya tiene un endpoint. Se reutiliza '{VS_ENDPOINT}' "
        f"en lugar de crear '{PREFERRED_ENDPOINT}'."
    )
else:
    VS_ENDPOINT = PREFERRED_ENDPOINT
    print(f"Creando endpoint STANDARD '{VS_ENDPOINT}'...")
    client.create_endpoint_and_wait(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD",
        verbose=True,
    )
    print("Endpoint creado y ONLINE.")

print(f"Endpoint en uso: {VS_ENDPOINT}")


## 5. Crear el índice Delta Sync

`columns_to_sync=["title"]` evita sincronizar `question` y `expected_response`. La clave
primaria y la columna de embeddings se incluyen siempre, por lo que el índice contiene
solamente `chunk_id`, `chunk_text` y `title`.

Si el índice ya existe, se dispara `sync()` porque su pipeline es `TRIGGERED`.


In [ ]:
if client.index_exists(index_name=INDEX_NAME):
    print(f"El índice ya existe: {INDEX_NAME}")
    index = client.get_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
    )
    existing_desc = index.describe()
    existing_spec = existing_desc.get("delta_sync_index_spec", {})
    existing_source = existing_spec.get("source_table")
    if existing_source and existing_source != T_CORPUS:
        raise RuntimeError(
            f"El índice existente apunta a {existing_source}, no a {T_CORPUS}. "
            "Usa otro nombre de índice o corrige el recurso desde Catalog Explorer."
        )
    print("Disparando sincronización incremental...")
    index.sync()
else:
    print(f"Creando índice Delta Sync: {INDEX_NAME}")
    index = client.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=T_CORPUS,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["title"],
    )
    print("Solicitud de creación enviada.")


## 6. Esperar hasta `ready` / `ONLINE`

La creación es asíncrona. El ciclo muestra cambios de estado, detecta fallos y deja un
timeout claro. Para 100 contextos suele ser pequeño, pero el aprovisionamiento del endpoint
puede tardar varios minutos.


In [ ]:
deadline = time.time() + TIMEOUT_MIN * 60
last_snapshot = None

while time.time() < deadline:
    desc = index.describe()
    status = desc.get("status", {})
    state = str(
        status.get("detailed_state")
        or status.get("state")
        or status.get("status")
        or "UNKNOWN"
    )
    ready = bool(status.get("ready", False))
    indexed = status.get("indexed_row_count", "?")
    snapshot = (state, ready, indexed)

    if snapshot != last_snapshot:
        print(
            f"[{time.strftime('%H:%M:%S')}] "
            f"state={state}, ready={ready}, filas_indexadas={indexed}"
        )
        last_snapshot = snapshot

    upper_state = state.upper()
    online = "ONLINE" in upper_state
    if ready and (online or upper_state == "UNKNOWN"):
        print("✓ Índice listo para consultas.")
        break
    if "FAIL" in upper_state:
        raise RuntimeError(f"Falló la construcción del índice: {status}")

    time.sleep(20)
else:
    raise TimeoutError(
        f"El índice no llegó a ready/ONLINE en {TIMEOUT_MIN} minutos. "
        "No lo borres: revisa Catalog Explorer y vuelve a ejecutar esta celda."
    )


## 7. Consulta semántica sin llamadas al LLM


In [ ]:
SEARCH_COLUMNS = ["chunk_id", "title", "chunk_text"]


def parse_search_results(result: dict) -> list[dict]:
    manifest = result.get("manifest", {}).get("columns", [])
    names = [c.get("name") for c in manifest if isinstance(c, dict)]
    rows = result.get("result", {}).get("data_array", [])
    if not names and rows:
        names = SEARCH_COLUMNS + ["score"]
    return [dict(zip(names, row)) for row in rows]


demo = (
    spark.table(T_CORPUS)
    .select("question", "chunk_id")
    .orderBy("question_id")
    .first()
)

search_result = index.similarity_search(
    query_text=demo["question"],
    columns=SEARCH_COLUMNS,
    num_results=3,
)
chunks = parse_search_results(search_result)

print(f"Pregunta: {demo['question']}")
print(f"Chunk gold: {demo['chunk_id']}\n")
for position, chunk in enumerate(chunks, start=1):
    print(
        f"{position}. {chunk.get('chunk_id')} "
        f"score={chunk.get('score', 'n/d')} title={chunk.get('title')}"
    )
    print(f"   {chunk.get('chunk_text', '')[:180]}...")

hit = demo["chunk_id"] in {c.get("chunk_id") for c in chunks}
print(f"\nHit@3 para el ejemplo: {int(hit)}")


## 8. Verificación en la UI

En **Catalog Explorer**:

1. abre el índice `agenteval_squadv2_index`;
2. confirma tipo **Delta Sync**, estado **ONLINE** y modo **Triggered**;
3. confirma `chunk_id` como Primary key;
4. confirma `chunk_text` y `databricks-gte-large-en`;
5. observa que `expected_response` no forma parte del índice.

En ejecuciones posteriores, el notebook reutiliza el endpoint y sincroniza el índice. No
crea recursos duplicados.

Siguiente notebook: **55 — Agente RAG instrumentado con MLflow**.
